# Sprint 4 — Hyperparameter Tuning — Recall First

Este notebook optimiza los mejores modelos baseline identificados previamente:

- DecisionTree
- LinearSVM
- LogisticRegression

La selección prioriza:

1. **Recall**, para detectar la mayor cantidad posible de `IsBadBuy = 1`.
2. **F2**, como métrica secundaria con mayor peso en recall.
3. **Precision**, como control para no marcar demasiados autos buenos como malos.

Además del tuning de hiperparámetros, se aplica **threshold tuning** con predicciones out-of-fold para no depender del umbral default `0.50`.

Los modelos avanzados de boosting, como **XGBoost** y **LightGBM**, se evalúan en el siguiente notebook.

In [4]:
from pathlib import Path
import sys
import json
import warnings

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from src.config import *
from src.io_utils import load_kick_data
from src.preprocessing import prepare_features, split_X_y
from src.models import get_cv
from src.tuning import randomized_tune
from src.evaluation import threshold_tuning_cv, select_best_threshold, get_oof_scores
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

N_ITER = 15
N_JOBS = -1 # Te recomiendo -1 para que use todos tus procesadores y sea más rápido
CANDIDATE_MODELS = [
    "HistGradientBoosting", 
    "GradientBoosting",
    "LogisticRegression"
]
cv = get_cv()

In [5]:
# Cargar muestra si existe; si no, cargar raw y muestrear estratificado.
sample_path = PROCESSED_DIR / "train_sample.csv"

if sample_path.exists():
    df_model = pd.read_csv(sample_path)
else:
    df_model = load_kick_data()
    df_model = df_model.dropna(subset=[TARGET]).copy()

df_model = prepare_features(df_model)
X, y = split_X_y(df_model)

if len(X) > SAMPLE_SIZE:
    X_sample, _, y_sample, _ = train_test_split(
        X,
        y,
        train_size=SAMPLE_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
else:
    X_sample, y_sample = X.copy(), y.copy()

print("X_sample:", X_sample.shape)
display(y_sample.value_counts().rename("count").to_frame())
display(y_sample.value_counts(normalize=True).rename("share").to_frame())

X_sample: (20000, 38)


,count
IsBadBuy,
0,17540
1,2460


,share
IsBadBuy,
0,0.877
1,0.123


In [6]:
# Tunear por recall -> F2 -> precision usando src.tuning.recall_refit dentro de randomized_tune().
tuning_rows = []
searches = {}

for model_name in CANDIDATE_MODELS:
    row, search = randomized_tune(
        model_name,
        X_sample,
        y_sample,
        n_iter=N_ITER,
        n_jobs=N_JOBS,
        cv=cv,
    )
    tuning_rows.append(row)
    searches[model_name] = search

tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values(["best_recall", "best_f2", "best_precision"], ascending=[False, False, False])
    .reset_index(drop=True)
)

tuning_path = REPORTS_DIR / "tuning_results_recall_first.csv"
tuning_results.to_csv(tuning_path, index=False)

display(tuning_results[[
    "model", "best_recall", "best_f2", "best_precision",
    "train_recall", "recall_gap_train_minus_cv", "seconds", "path"
]])
print("Guardado:", tuning_path)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Fitting 5 folds for each of 15 candidates, totalling 75 fits


,model,best_recall,best_f2,best_precision,train_recall,recall_gap_train_minus_cv,seconds,path
0,LogisticRegression,0.590244,0.472431,0.262738,0.636484,0.04624,31.0,c:\Users\PONCE\dp261-g1\models\tuned_LogisticRegression.pkl
1,HistGradientBoosting,0.249593,0.289709,0.813438,0.275203,0.02561,256.5,c:\Users\PONCE\dp261-g1\models\tuned_HistGradientBoosting.pkl
2,GradientBoosting,0.249593,0.289709,0.813438,0.275203,0.02561,257.4,c:\Users\PONCE\dp261-g1\models\tuned_GradientBoosting.pkl


Guardado: c:\Users\PONCE\dp261-g1\reports\tuning_results_recall_first.csv


In [8]:
# --- CELDA NUEVA: OPTUNA PARA GRADIENT BOOSTING ---
from src.tuning import optuna_tune_gradient_boosting

print("Iniciando optimización avanzada con Optuna para GradientBoosting...")
# n_trials=30 es un buen equilibrio entre tiempo y calidad
study, best_gb_model, gb_path = optuna_tune_gradient_boosting(X_sample, y_sample, n_trials=30)

print(f"✅ ¡Optimización terminada!")
print(f"Mejores parámetros: {study.best_params}")

# 1. Creamos un objeto 'dummy' para que sea compatible con tu lógica anterior de searches
class MockSearch:
    def __init__(self, estimator):
        self.best_estimator_ = estimator

# Agregamos al diccionario para el Threshold Tuning
searches["GradientBoosting_Optuna"] = MockSearch(best_gb_model)

# 2. Integramos el resultado en la tabla tuning_results para el Ranking Final
new_row = pd.DataFrame([{
    "model": "GradientBoosting_Optuna",
    "best_recall": study.best_value, 
    "best_f2": 0,          # Placeholder para que el DataFrame sea consistente
    "best_precision": 0,   # Placeholder
    "path": str(gb_path)
}])

tuning_results = pd.concat([tuning_results, new_row], ignore_index=True)

print("✅ Modelo de Optuna integrado en la tabla de resultados y listo para Threshold Tuning.")

Iniciando optimización avanzada con Optuna para GradientBoosting...


[I 2026-05-28 16:59:19,429] A new study created in memory with name: no-name-808e4ae5-91e7-49ce-b26b-de4e9613bb42
[I 2026-05-28 17:00:34,237] Trial 0 finished with value: 0.2482568616976361 and parameters: {'n_estimators': 157, 'learning_rate': 0.11180190486692741, 'max_depth': 4, 'min_samples_leaf': 44, 'subsample': 0.9611552759404249}. Best is trial 0 with value: 0.2482568616976361.
[I 2026-05-28 17:01:48,091] Trial 1 finished with value: 0.2486628210737051 and parameters: {'n_estimators': 295, 'learning_rate': 0.05516879558530038, 'max_depth': 2, 'min_samples_leaf': 17, 'subsample': 0.9933545385490361}. Best is trial 1 with value: 0.2486628210737051.
[I 2026-05-28 17:02:51,086] Trial 2 finished with value: 0.24377947833376723 and parameters: {'n_estimators': 256, 'learning_rate': 0.030590971686425935, 'max_depth': 2, 'min_samples_leaf': 46, 'subsample': 0.8978800359178265}. Best is trial 1 with value: 0.2486628210737051.
[I 2026-05-28 17:03:09,558] Trial 3 finished with value: 0.236

✅ ¡Optimización terminada!
Mejores parámetros: {'n_estimators': 275, 'learning_rate': 0.15574982303190235, 'max_depth': 4, 'min_samples_leaf': 22, 'subsample': 0.6529420106394438}
✅ Modelo de Optuna integrado en la tabla de resultados y listo para Threshold Tuning.


In [10]:
# Buscar threshold con predicciones out-of-fold; no usar threshold default 0.50.
threshold_frames = []

for model_name, search in searches.items():
    results = threshold_tuning_cv(
        model_name,
        search.best_estimator_,
        X_sample,
        y_sample,
        cv=cv,
        n_jobs=N_JOBS,
    )
    threshold_frames.append(results)

threshold_results = pd.concat(threshold_frames, ignore_index=True)

threshold_results_path = REPORTS_DIR / "threshold_tuning_all_results_recall_first.csv"
threshold_results.to_csv(threshold_results_path, index=False)

# Definimos una función de selección con criterio de negocio
def select_balanced_threshold(df_results, min_precision=0.20):
    # Filtramos solo aquellos umbrales que no destruyen el negocio
    valid_points = df_results[df_results["precision"] >= min_precision]
    
    if valid_points.empty:
        # Si ninguno llega al 20%, bajamos un poco la vara al 15%
        valid_points = df_results[df_results["precision"] >= 0.15]
    
    # De los puntos válidos, elegimos el que maximice F2 (que ya prioriza Recall)
    if not valid_points.empty:
        return valid_points.loc[valid_points["f2"].idxmax()]
    else:
        return df_results.loc[df_results["f2"].idxmax()]

# Aplicamos la nueva lógica corrigiendo el manejo de índices
best_thresholds = (
    threshold_results
    .groupby("model", group_keys=False)
    .apply(select_balanced_threshold)
)

# Si Pandas puso el modelo como índice, lo recuperamos como columna
if "model" not in best_thresholds.columns:
    best_thresholds = best_thresholds.reset_index()

# Ahora nos aseguramos de que no haya duplicados y el índice esté limpio
best_thresholds = best_thresholds.reset_index(drop=True)

# Guardamos los resultados
best_thresholds_path = REPORTS_DIR / "threshold_tuning_best_recall_first.csv"
best_thresholds.to_csv(best_thresholds_path, index=False)

# Mostramos la tabla final
display(best_thresholds[[
    "model", "threshold", "recall", "f2", "precision",
    "positive_rate", "tp", "fp", "fn", "tn"
]])
print("Guardado:", threshold_results_path)
print("Guardado:", best_thresholds_path)

,model,threshold,recall,f2,precision,positive_rate,tp,fp,fn,tn
0,GradientBoosting,0.09,0.742276,0.494315,0.211587,0.4315,1826,6804,634,10736
1,GradientBoosting_Optuna,0.09,0.672764,0.485565,0.229797,0.3601,1655,5547,805,11993
2,HistGradientBoosting,0.09,0.742276,0.494315,0.211587,0.4315,1826,6804,634,10736
3,LogisticRegression,0.45,0.674390,0.484691,0.228073,0.3637,1659,5615,801,11925


Guardado: c:\Users\PONCE\dp261-g1\reports\threshold_tuning_all_results_recall_first.csv
Guardado: c:\Users\PONCE\dp261-g1\reports\threshold_tuning_best_recall_first.csv


In [11]:
# Ranking final: hiperparámetros + mejor threshold.
final_ranking = tuning_results.merge(
    best_thresholds.add_prefix("threshold_"),
    left_on="model",
    right_on="threshold_model",
    how="left",
)

final_ranking = (
    final_ranking
    .sort_values(["threshold_recall", "threshold_f2", "threshold_precision"], ascending=[False, False, False])
    .reset_index(drop=True)
)

final_ranking_path = REPORTS_DIR / "final_model_ranking_recall_first_with_threshold.csv"
final_ranking.to_csv(final_ranking_path, index=False)

display(final_ranking[[
    "model", "best_recall", "best_f2", "best_precision",
    "threshold_threshold", "threshold_recall", "threshold_f2", "threshold_precision",
    "threshold_positive_rate", "threshold_tp", "threshold_fp", "threshold_fn", "threshold_tn",
    "path"
]])
print("Guardado:", final_ranking_path)

,model,best_recall,best_f2,best_precision,threshold_threshold,threshold_recall,threshold_f2,threshold_precision,threshold_positive_rate,threshold_tp,threshold_fp,threshold_fn,threshold_tn,path
0,HistGradientBoosting,0.249593,0.289709,0.813438,0.09,0.742276,0.494315,0.211587,0.4315,1826,6804,634,10736,c:\Users\PONCE\dp261-g1\models\tuned_HistGradientBoosting.pkl
1,GradientBoosting,0.249593,0.289709,0.813438,0.09,0.742276,0.494315,0.211587,0.4315,1826,6804,634,10736,c:\Users\PONCE\dp261-g1\models\tuned_GradientBoosting.pkl
2,LogisticRegression,0.590244,0.472431,0.262738,0.45,0.674390,0.484691,0.228073,0.3637,1659,5615,801,11925,c:\Users\PONCE\dp261-g1\models\tuned_LogisticRegression.pkl
3,GradientBoosting_Optuna,0.266973,0.000000,0.000000,0.09,0.672764,0.485565,0.229797,0.3601,1655,5547,805,11993,c:\Users\PONCE\dp261-g1\models\tuned_gradient_boosting_optuna.pkl


Guardado: c:\Users\PONCE\dp261-g1\reports\final_model_ranking_recall_first_with_threshold.csv


In [12]:
# Revisar matriz de confusión del mejor modelo final.
best_final = final_ranking.iloc[0]
best_model_name = best_final["model"]
best_threshold = float(best_final["threshold_threshold"])
best_estimator = searches[best_model_name].best_estimator_

scores, score_type = get_oof_scores(
    best_estimator,
    X_sample,
    y_sample,
    cv=cv,
    n_jobs=N_JOBS,
)

y_pred = (scores >= best_threshold).astype(int)
cm = confusion_matrix(y_sample, y_pred, labels=[0, 1])

print("Mejor modelo:", best_model_name)
print("Threshold:", best_threshold)
print("Score type:", score_type)
display(pd.DataFrame(cm, index=["real_0", "real_1"], columns=["pred_0", "pred_1"]))
print(classification_report(y_sample, y_pred, zero_division=0))

Mejor modelo: HistGradientBoosting
Threshold: 0.09
Score type: predict_proba


,pred_0,pred_1
real_0,10736,6804
real_1,634,1826


              precision    recall  f1-score   support

           0       0.94      0.61      0.74     17540
           1       0.21      0.74      0.33      2460

    accuracy                           0.63     20000
   macro avg       0.58      0.68      0.54     20000
weighted avg       0.85      0.63      0.69     20000



In [13]:
# Guardar metadata del modelo final.
final_metadata = {
    "model_name": best_model_name,
    "threshold": best_threshold,
    "score_type": score_type,
    "priority": "recall_then_f2_then_precision",
    "model_path": best_final["path"],
    "ranking_path": str(final_ranking_path),
}

metadata_path = MODELS_DIR / "final_recall_first_model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(final_metadata, f, indent=2, ensure_ascii=False)

print("Metadata guardada:", metadata_path)
final_metadata

Metadata guardada: c:\Users\PONCE\dp261-g1\models\final_recall_first_model_metadata.json


{'model_name': 'HistGradientBoosting',
 'threshold': 0.09,
 'score_type': 'predict_proba',
 'priority': 'recall_then_f2_then_precision',
 'model_path': 'c:\\Users\\PONCE\\dp261-g1\\models\\tuned_HistGradientBoosting.pkl',
 'ranking_path': 'c:\\Users\\PONCE\\dp261-g1\\reports\\final_model_ranking_recall_first_with_threshold.csv'}